In [39]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.models import vgg19
import numpy as np
import os 
import cv2
from PIL import Image
import matplotlib.pyplot as plt
%matplotlib inline
import random
from tqdm import tqdm
import math

In [41]:
# Set Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


# Swin Transformer Components

In [44]:
# Swin Transformer Components

class PatchEmbed(nn.Module):
    """Image to Patch Embedding"""
    def __init__(self, img_size=128, patch_size=4, in_chans=3, embed_dim=96):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.patches_resolution = img_size // patch_size
        self.num_patches = self.patches_resolution ** 2
        
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        
    def forward(self, x):
        B, C, H, W = x.shape
        x = self.proj(x).flatten(2).transpose(1, 2)  # B Ph*Pw C
        return x

class PatchMerging(nn.Module):
    """Patch Merging Layer"""
    def __init__(self, input_resolution, dim):
        super().__init__()
        self.input_resolution = input_resolution
        self.dim = dim
        self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)
        self.norm = nn.LayerNorm(4 * dim)
        
    def forward(self, x):
        H, W = self.input_resolution
        B, L, C = x.shape
        assert L == H * W, "input feature has wrong size"
        assert H % 2 == 0 and W % 2 == 0, f"x size ({H}*{W}) are not even."
        
        x = x.view(B, H, W, C)
        
        x0 = x[:, 0::2, 0::2, :]  # B H/2 W/2 C
        x1 = x[:, 1::2, 0::2, :]  # B H/2 W/2 C
        x2 = x[:, 0::2, 1::2, :]  # B H/2 W/2 C
        x3 = x[:, 1::2, 1::2, :]  # B H/2 W/2 C
        x = torch.cat([x0, x1, x2, x3], -1)  # B H/2 W/2 4*C
        x = x.view(B, -1, 4 * C)  # B H/2*W/2 4*C
        
        x = self.norm(x)
        x = self.reduction(x)
        
        return x

class WindowAttention(nn.Module):
    """Window based multi-head self attention"""
    def __init__(self, dim, window_size, num_heads, qkv_bias=True, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5
        
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        
    def forward(self, x):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        q = q * self.scale
        attn = (q @ k.transpose(-2, -1))
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        
        x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

class SwinTransformerBlock(nn.Module):
    """Swin Transformer Block"""
    def __init__(self, dim, input_resolution, num_heads, window_size=7,
                 shift_size=0, mlp_ratio=4., qkv_bias=True, drop=0., attn_drop=0.):
        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.mlp_ratio = mlp_ratio
        
        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(
            dim, window_size=(self.window_size, self.window_size), num_heads=num_heads,
            qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=drop)
        
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(mlp_hidden_dim, dim),
            nn.Dropout(drop)
        )
        
    def forward(self, x):
        H, W = self.input_resolution
        B, L, C = x.shape
        assert L == H * W, "input feature has wrong size"
        
        shortcut = x
        x = self.norm1(x)
        x = x.view(B, H, W, C)
        
        # Window partition
        x_windows = self.window_partition(x, self.window_size)
        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)
        
        # W-MSA/SW-MSA
        attn_windows = self.attn(x_windows)
        
        # Merge windows
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)
        x = self.window_reverse(attn_windows, self.window_size, H, W)
        x = x.view(B, H * W, C)
        
        # FFN
        x = shortcut + x
        x = x + self.mlp(self.norm2(x))
        
        return x
    
    def window_partition(self, x, window_size):
        B, H, W, C = x.shape
        x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
        windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
        return windows
    
    def window_reverse(self, windows, window_size, H, W):
        B = int(windows.shape[0] / (H * W / window_size / window_size))
        x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
        x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
        return x

# LPDGAN Components

In [63]:
class LatentFusionModule(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.conv_aplha = nn.Sequential(
            nn.Conv2d(dim, dim // 2, 3, 1, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(dim // 2, dim, 3, 1, 1)
        )
        self.conv_beta = nn.Sequential(
            nn.Conv2d(dim, dim // 2, 3, 1, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(dim // 2, dim, 3, 1, 1)
        )

    def forward(self, x_low, x_high):
        x_high_1, x_high_2 = torch.chunk(x_high, 2, dim=1)
        alpha = self.conv_alpha(x_high_1)
        beta = self.conv_beta(x_high_2)
        modulatetd = alpha * x_low + beta
        fused = torch.cad([modulated, x_high], dim=1)
        return fused

class SwinEncoder(nn.Module):
    def __init__(self, img_size=128, patch_size=4, embed_dim=96, depths=[2, 2, 6, 2],
                 num_heads=[3, 6, 12, 24], window_size=7):
        super().__init__()
        self.embed_dim = embed_dim
        self.depths = depths
        self.num_layers = len(depths)

        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, in_chans=3, embed_dim=embed_dim)

        self.layers = nn.ModuleList()
        for i_layer in range(self.num_layers):
            layer = nn.ModuleList([
                SwinTransformerBlock(
                    dim=int(embed_dim * 2 ** i_layer),
                    input_resolution=(img_size // (patch_size * 2 ** i_layer),
                                      img_size // (patch_size * 2 ** i_layer) * 2),
                    num_heads=num_heads[i_layer],
                    window_size=window_size,
                    shift_size=0 if (j % 2 == 0) else window_size // 2)
                for j in range(depths[i_layer])
            ])
            self.layers.append(layer)

            if i_layer < self.num_layers - 1:
                self.layers.append(PatchMerging(
                    input_resolution=(img_size // (patch_size * 2 ** i_layer),
                                      img_size // (patch_size * 2 ** i_layer) * 2),
                    dim=int(embed_dim * 2 ** i_layer)))

        self.norm = nn.LayerNorm(int(embed_dim * 2 ** (self.num_layers - 1)))

    def forward(self, x):
        x = self.patch_embed(x)
        features = []

        layer_idx = 0
        for i_layer in range(self.num_layers):
            for block in self.layers[layer_idx]:
                x = block(x)
            layer_idx += 1

            features.append(x)

            if i_layer < self.num_layers - 1:
                x = self.layers[layer_idx](x)
                layer_idx += 1

        x = self.norm(x)
        return features

class TextReconstructionModule(nn.Module):
    def __init__(self, feature_dim, hidden_dim=512, vocab_size=37, max_length=8):
        super().__init__()
        self.feature_dim = feature_dim
        self.hidden_dim = hidden_dim
        self.vocab_size = vocab_size
        self.max_length = max_length

        self.feature_fusion = nn.Conv2d(feature_dim * 2, feature_dim, 3, 1, 1)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.text_head = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, vocab_size * max_length)
        )

    def forward(self, decoder_feature, fusion_feature):
        fused = torch.cat([decoder_feature, fusion_feature], dim=1)
        fused = self.feature_fusion(fused)
        pooled = self.global_pool(fused)
        pooled = pooled.flatten(1)
        text_logits = self.text_head(pooled)
        text_logits = text_logits.view(-1, self.max_length, self.vocab_size)
        return text_logits

class Generator(nn.Module):
    def __init__(self, img_size=128):
        super().__init__()
        self.img_size = img_size

        self.encoder_1 = SwinEncoder(img_size=img_size, embed_dim=96)
        self.encoder_2 = SwinEncoder(img_size=img_size // 2, embed_dim=96)
        self.encoder_3 = SwinEncoder(img_size=img_size // 4, embed_dim=96)

        self.fusion_1_2 = LatentFusionModule(dim=96)
        self.fusion_2_3 = LatentFusionModule(dim=192)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(384, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 32, 3, 1, 1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 3, 3, 1, 1),
            nn.Tanh()
        )

        self.output_conv_1 = nn.Conv2d(256, 3, 3, 1, 1)
        self.output_conv_2 = nn.Conv2d(128, 3, 3, 1, 1)

        self.text_module = TextReconstructionModule(feature_dim=128)

    def forward(self, x):
        batch_size = x.size(0)

        x1 = x
        x2 = F.interpolate(x, scale_factor=0.5, mode='bilinear', align_corners=False)
        x3 = F.interpolate(x, scale_factor=0.25, mode='bilinear', align_corners=False)

        features_1 = self.encoder_1(x1)
        features_2 = self.encoder_2(x2)
        features_3 = self.encoder_3(x3)

        feat_1 = features_1[-1].view(batch_size, -1, self.img_size // 16, self.img_size // 8)
        feat_2 = features_2[-1].view(batch_size, -1, self.img_size // 32, self.img_size // 16)
        feat_3 = features_3[-1].view(batch_size, -1, self.img_size // 64, self.img_size // 32)

        feat_2_up = F.interpolate(feat_2, size=feat_1.shape[2:], mode='bilinear', align_corners=False)
        feat_3_up = F.interpolate(feat_3, size=feat_2.shape[2:], mode='bilinear', align_corners=False)

        fused_1_2 = self.fusion_1_2(feat_1, feat_2_up)
        fused_2_3 = self.fusion_2_3(feat_2, feat_3_up)

        final_feat = torch.cat([fused_1_2,
                               F.interpolate(fused_2_3, size=fused_1_2.shape[2:], mode='bilinear', align_corners=False)],
                               dim=1)

        decoder_features = []
        x_out = final_feat

        for i, layer in enumerate(self.decoder):
            x_out = layer(x_out)
            if i in [1, 4]:
                decoder_features.append(x_out)

        output_1 = x_out
        output_2 = F.interpolate(self.output_conv_2(decoder_features[1]),
                                size=output_1.shape[2:], mode='bilinear', align_corners=False)
        output_3 = F.interpolate(self.output_conv_1(decoder_features[0]),
                                size=output_1.shape[2:], mode='bilinear', align_corners=False)

        text_logits = self.text_module(decoder_features[1],
                                      F.interpolate(fused_2_3, size=decoder_features[1].shape[2:],
                                                   mode='bilinear', align_corners=False))

        return {
            'output_1': output_1,
            'output_2': output_2,
            'output_3': output_3,
            'text_logits': text_logits
        }

class GlobalDiscriminator(nn.Module):
    """Global Discriminator for overall image quality"""
    def __init__(self, input_channels=3):
        super().__init__()

        self.model = nn.Sequential(
            nn.Conv2d(input_channels, 64, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 256, 4, 2, 1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(256, 512, 4, 2, 1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(512, 1, 4, 2, 1)
        )

    def forward(self, x):
        return self.model(x)

class PartitionDiscriminator(nn.Module):
    """Partition Discriminator for character-level discrimination"""
    def __init__(self, input_channels=3, partition_size=32):
        super().__init__()
        self.partition_size = partition_size

        self.model = nn.Sequential(
            nn.Conv2d(input_channels, 32, 3, 1, 1),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(32, 64, 4, 2, 1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 1, 4, 2, 1)
        )

    def forward(self, x):
        batch_size, _, height, width = x.shape
        partitions = []
        num_partitions = 3

        for _ in range(num_partitions):
            start_h = random.randint(0, max(0, height - self.partition_size))
            start_w = random.randint(0, max(0, width - self.partition_size))

            partition = x[:, :, start_h:start_h+self.partition_size, 
                         start_w:start_w+self.partition_size]

            if partition.size(2) < self.partition_size or partition.size(3) < self.partition_size:
                partition = F.interpolate(partition, size=(self.partition_size, self.partition_size),
                                          mode='bilinear', align_corners=False)

            partitions.append(self.model(partition))

        return torch.mean(torch.stack(partitions), dim=0)

In [65]:
import sys
ROOT_DIR = os.path.join('..', '..', '..')
sys.path.append(os.path.join(ROOT_DIR, 'src', 'dataset'))
from license_plate_dataset import LicensePlateDataset

In [67]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

batch_size = 16

train_blur_dir = os.path.join(ROOT_DIR, 'data', 'h_blur', 'train', 'h_blur')
train_sharp_dir = os.path.join(ROOT_DIR, 'data', 'h_blur', 'train', 'enhanced')

valid_blur_dir = os.path.join(ROOT_DIR, 'data', 'h_blur', 'valid', 'h_blur')
valid_sharp_dir = os.path.join(ROOT_DIR, 'data', 'h_blur', 'valid', 'enhanced')

test_blur_dir = os.path.join(ROOT_DIR, 'data', 'h_blur', 'test', 'h_blur')

train_ds = LicensePlateDataset(train_blur_dir, train_sharp_dir, transform=transform, img_size=(128, 256))
valid_ds = LicensePlateDataset(valid_blur_dir, valid_sharp_dir, transform=transform, img_size=(128, 256))

train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4)
valid_dl = DataLoader(train_ds, batch_size=batch_size*2, shuffle=True, num_workers=4)

## Loss Functions

In [70]:
class VGGPerceptualLoss(nn.Module):
    """VGG Perceptual Loss"""
    def __init__(self):
        super().__init__()
        vgg = vgg19(pretrained=True).features
        self.feature_layers = nn.ModuleList([
            vgg[:8],
            vgg[:17],
            vgg[:26],
            vgg[:35]
        ])

        for param in self.parameters():
            param.requires_grad = False

    def forward(self, pred, target):
        loss = 0
        for layer in self.feature_layers:
            pred_feat = layer(pred)
            target_feat = layer(target)
            loss += F.mse_loss(pred_feat, target_feat)
        return loss

class TextLoss(nn.Module):
    """Text Reconstruction Loss"""
    def __init__(self):
        super().__init__()
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, pred_text, target_text):
        batch_size, seq_len, vocab_size = pred_text.shape
        dummy_target = torch.zeros(batch_size, seq_len, dtype=torch.long, device=pred_text.device)

        loss = 0
        for i in range(seq_len):
            loss += self.criterion(pred_text[:, i, :], dummy_target[:, i])

        return loss / seq_len

def gradient_penalty(discriminator, real_samples, fake_samples, device):
    """Calculate gradient penalty for WGAN-GP"""
    batch_size = real_samples.size(0)
    alpha = torch.rand(batch_size, 1, 1, 1, device=device)

    interpolates = alpha * real_samples + (1 - alpha) * fake_samples
    interpolates.requires_grad(True)

    d_interpolates = discriminator(interpolates)

    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=torch.ones_like(d_interpolates),
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]

    gradients = gradients.view(batch_size, -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()

    return gradient_penalty

# Training Function

In [73]:
def display_images(images, title, nrow=4):
    """Helper function to display images using Matplotlib"""
    # Denormalize images from [-1, 1] or [0, 1] to [0, 1]
    images = (images + 1) / 2 if images.min() < 0 else images
    grid = make_grid(images, nrow=nrow, normalize=False)
    plt.figure(figsize=(15, 5))
    plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
    plt.title(title)
    plt.axis('off')
    plt.show()

In [83]:
def train(generator, global_dics, partition_disc, train_dl, num_epochs=100,
          lr=0.0002, device=device):

    # Optimizers
    g_optimizer = optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
    d_global_optimizer = optim.Adam(global_disc.parameters(), lr=lr, betas=(0.5, 0.999))
    d_partition_optimizer = optim.Adam(partition_disc.parameters(), lr=lr, betas=(0.5, 0.999))

    # Loss functions
    l1_loss = nn.L1Loss()
    perceptual_loss = VGGPerceptualLoss().to(device)
    text_loss = TextLoss()

    # Loss weights
    lambda_l1 = 100.0
    lambda_perceptual = 10.0
    lambda_text = 10.0
    lambda_adv_global = 1.0
    lambda_adv_partition = 1.0
    lambda_gp = 10.0

    generator.train()
    global_disc.train()
    partition_disc.train()

    # Training history
    g_losses = []
    d_losses = []

    for epoch in range(num_epochs):
        epoch_g_loss = 0.0
        epoch_d_loss = 0.0

        for batch_idx, (blur_imgs, sharp_imgs) in enumerate(tqdm(train_dl, desc=f"Epoch {epoch+1}/{num_epochs}")):
            batch_size = blur_imgs.size(0)
            blur_imgs = blur_imgs.to(device)
            sharp_imgs = sharp_imgs.to(device)

            # ==================== Train Discriminators ====================

            # Global Discriminator
            d_global_optimizer.zero_grad()

            with torch.no_grad():
                gen_outputs = generator(blur_imgs)
                fake_imgs = gen_outputs['output_1']

            real_global = global_disc(sharp_imgs).mean()
            fake_global = global_disc(fake_imgs).mean()
            gp_global = gradient_penalty(global_disc, sharp_imgs, fake_imgs, device)

            d_global_loss = -real_global + fake_global + lambda_gp * gp_global
            d_global_loss.backward()
            d_global_optimizer.step()

            # Partition Discriminator
            d_partition_optimizer.zero_grad()

            real_partition = partition_disc(sharp_imgs).mean()
            fake_partition = partition_disc(fake_imgs).mean()
            gp_partition = gradient_penalty(partition_disc, sharp_imgs, fake_imgs, device)

            d_partition_loss = -real_partition + fake_partition + lambda_gp * gp_partition
            d_partition_loss.backward()
            d_partition_optimizer.step()

            d_loss = d_global_loss + d_partition_loss
            epoch_d_loss += d_loss.item()

            # ==================== Train Generator ====================
            g_optimizer.zero_grad()

            gen_outputs = generator(blur_imgs)
            fake_imgs_1 = gen_outputs['output_1']
            fake_imgs_2 = gen_outputs['output_2']
            fake_imgs_3 = gen_outputs['output_3']
            text_logits = gen_outputs['text_logits']

            # Adversarial Losses
            adv_global_loss = -global_disc(fake_imgs_1).mean()
            adv_partition_loss = -partition_disc(fake_imgs_1).mean()

            # Reconstruction Losses
            l1_loss_1 = l1_loss(fake_imgs_1, sharp_imgs)
            l1_loss_2 = l1_loss(fake_imgs_2, sharp_imgs)
            l1_loss_3 = l1_loss(fake_imgs_3, sharp_imgs)

            perceptual_loss_1 = perceptual_loss(fake_imgs_1, sharp_imgs)
            perceptual_loss_2 = perceptual_loss(fake_imgs_2, sharp_imgs)
            perceptual_loss_3 = perceptual_loss(fake_imgs_3, sharp_imgs)

            text_loss_val = text_loss(text_logits, None) # Note: Using dummy target for now

            # Combined Generator Loss
            g_loss = (lambda_adv_global * adv_global_loss + 
                     lambda_adv_partition * adv_partition_loss +
                     lambda_l1 * (l1_loss_1 + l1_loss_2 + l1_loss_3) +
                     lambda_perceptual * (perceptual_loss_1 + perceptual_loss_2 + perceptual_loss_3) +
                     lambda_text * text_loss_val)

            g_loss.backward()
            g_optimizer.step()

            epoch_g_loss += g_loss.item()

        # Log epoch losses
        adv_g_loss = epoch_g_loss / len(train_dl)
        adv_d_loss = epoch_d_loss / len(train_dl)
        g_losses.append(avg_g_loss)
        d_losses.append(avg_d_loss)

        print(f"Epoch [{epoch+1}/{num_epochs}] G Loss: {avg_g_loss:.4f}, D Loss: {avg_d_loss:.4f}")

        # Display generated images from train_dl
        with torch.no_grad():
            generator.eval()
            # Use the last batch from train_dl
            gen_outputs = generator(blur_imgs[:4])  # Limit to 4 images
            fake_imgs = gen_outputs['output_1']
            # Concatenate blurry, generated, sharp for display
            display_images(torch.cat([blur_imgs[:4], fake_imgs, sharp_imgs[:4]]), 
                           f"Epoch {epoch+1} Train: Blurry | Generated | Sharp")
            generator.train()

        # Generate and display 2 images from valid_dl and test_dl
        with torch.no_grad():
            generator.eval()
            # Valid_dl images
            valid_iter = iter(valid_dl)
            try:
                blur_valid, sharp_valid, _ = next(valid_dl)
                blur_valid = blur_valid.to(device)
                sharp_valid = sharp_valid.to(device)
                gen_outputs = generator(blur_valid[:2])  # 2 images
                fake_valid = gen_outputs['output_1']
                display_images(torch.cat([blur_valid[:2], fake_valid, sharp_valid[:2]]), 
                               f"Epoch {epoch+1} Valid: Blurry | Generated | Sharp")
            except StopIteration:
                print("Valid dataloader empty, skipping.")
            
            # Test_dl images
            test_iter = iter(test_dl)
            try:
                blur_test, sharp_test, _ = next(test_dl)
                blur_test = blur_test.to(device)
                sharp_test = sharp_test.to(device)
                gen_outputs = generator(blur_test[:2])  # 2 images
                fake_test = gen_outputs['output_1']
                display_images(torch.cat([blur_test[:2], fake_test, sharp_test[:2]]), 
                               f"Epoch {epoch+1} Test: Blurry | Generated | Sharp")
            except StopIteration:
                print("Test dataloader empty, skipping.")
            generator.train()
        
        # Save model checkpoints
        if (epoch + 1) % 10 == 0:
            torch.save(generator.state_dict(), f'generator_epoch_{epoch+1}.pth')
            torch.save(global_disc.state_dict(), f'global_disc_epoch_{epoch+1}.pth')
            torch.save(partition_disc.state_dict(), f'partition_disc_epoch_{epoch+1}.pth')

    return g_losses, d_losses

In [85]:
generator = Generator(img_size=128).to(device)
global_disc = GlobalDiscriminator().to(device)
partition_disc = PartitionDiscriminator().to(device)

# Train the model
g_losses, d_losses = train(generator, global_disc, partition_disc, train_dl, num_epochs=100, lr=0.0002, device=device)

D:\User\Jansen\Self Study\2025 - 05 - MAY\LiPAD\lipad-venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
D:\User\Jansen\Self Study\2025 - 05 - MAY\LiPAD\lipad-venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Epoch 1/100:   0%|                                                                             | 0/743 [00:26<?, ?it/s]


RuntimeError: shape '[16, 4, 7, 9, 7, 96]' is invalid for input of size 3145728

In [79]:
generator = Generator(img_size=128).to(device)
total_params = sum(p.numel() for p in generator.parameters())
total_params

86352753